JATIN BAJAJ
1024170389


In [2]:


!pip -q install nltk scikit-learn gradio seaborn 

import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# ============================================================
# 1. LABELED DATASET (64 QUERIES, 8 INTENTS)
# ============================================================

data = [
    # Symptoms
    ("What are common symptoms of diabetes?", "symptoms"),
    ("Why am I feeling thirsty frequently?", "symptoms"),
    ("Can diabetes cause blurred vision?", "symptoms"),
    ("What are signs of high blood sugar?", "symptoms"),
    ("Why do diabetic patients feel hungry?", "symptoms"),
    ("Can diabetes cause weight loss?", "symptoms"),
    ("What are symptoms of low blood sugar?", "symptoms"),
    ("Why do I feel tired with diabetes?", "symptoms"),

    # Glucose Monitoring
    ("How often should I check blood glucose?", "glucose_monitoring"),
    ("What is a glucose meter?", "glucose_monitoring"),
    ("When should I record sugar readings?", "glucose_monitoring"),
    ("What is fasting blood sugar?", "glucose_monitoring"),
    ("How do I use a glucose meter?", "glucose_monitoring"),
    ("Why should I maintain a sugar log?", "glucose_monitoring"),
    ("What does HbA1c measure?", "glucose_monitoring"),
    ("Should I share glucose readings with my doctor?", "glucose_monitoring"),

    # Medication
    ("What is insulin used for?", "medication"),
    ("Why are diabetes medicines prescribed?", "medication"),
    ("What if I forget my diabetes medicine?", "medication"),
    ("Can I stop taking diabetes medicines?", "medication"),
    ("What are side effects of diabetes medicines?", "medication"),
    ("Why do some patients need insulin?", "medication"),
    ("Can I change my insulin dose myself?", "medication"),
    ("How should I discuss medicine problems with my doctor?", "medication"),

    # Nutrition
    ("What foods are good for diabetes?", "nutrition"),
    ("Why should I control carbohydrate portions?", "nutrition"),
    ("Can diabetic patients drink sugary drinks?", "nutrition"),
    ("What is a balanced diabetic diet?", "nutrition"),
    ("Are whole grains good for diabetes?", "nutrition"),
    ("Why is fibre important?", "nutrition"),
    ("How should I plan meals?", "nutrition"),
    ("Can diabetic patients eat fruits?", "nutrition"),

    # Exercise
    ("What exercise is good for diabetes?", "exercise"),
    ("Can walking reduce blood sugar?", "exercise"),
    ("Should I monitor glucose during exercise?", "exercise"),
    ("Why should exercise be started gradually?", "exercise"),
    ("What precautions should diabetics take during workouts?", "exercise"),
    ("Is strength training useful for diabetes?", "exercise"),
    ("How often should I exercise?", "exercise"),
    ("Should I consult a doctor before intense exercise?", "exercise"),

    # Complications
    ("What are complications of diabetes?", "complications"),
    ("How does diabetes affect eyes?", "complications"),
    ("Why are foot checks important?", "complications"),
    ("Can diabetes damage kidneys?", "complications"),
    ("How does diabetes affect the heart?", "complications"),
    ("What is diabetic neuropathy?", "complications"),
    ("Why are eye screenings necessary?", "complications"),
    ("How can complications be prevented?", "complications"),

    # Emergency
    ("What if a diabetic patient becomes unconscious?", "emergency"),
    ("A diabetic patient is having a seizure.", "emergency"),
    ("My diabetic relative cannot swallow and is confused.", "emergency"),
    ("A diabetic person has chest pain and breathing difficulty.", "emergency"),
    ("A person with diabetes is unresponsive.", "emergency"),
    ("A diabetic patient has severe breathing difficulty.", "emergency"),
    ("My father collapsed and is not responding.", "emergency"),
    ("A diabetic patient has severe confusion.", "emergency"),

    # Prevention
    ("How can type 2 diabetes be prevented?", "prevention"),
    ("Does healthy weight reduce diabetes risk?", "prevention"),
    ("Can exercise prevent type 2 diabetes?", "prevention"),
    ("How does healthy food prevent diabetes?", "prevention"),
    ("Who should get diabetes screening?", "prevention"),
    ("Does family history increase diabetes risk?", "prevention"),
    ("What lifestyle habits improve metabolic health?", "prevention"),
    ("How can I discuss diabetes prevention with my doctor?", "prevention")
]

df = pd.DataFrame(data, columns=["query", "intent"])

print("Total queries:", len(df))
print("\nIntent distribution:")
print(df["intent"].value_counts())

# ============================================================
# 2. HIERARCHICAL TAXONOMY
# ============================================================

taxonomy = {
    "Understanding Diabetes": [
        "Symptoms",
        "Complications"
    ],
    "Monitoring and Treatment": [
        "Glucose Monitoring",
        "Medication"
    ],
    "Daily Self Management": [
        "Nutrition",
        "Exercise"
    ],
    "Risk Management": [
        "Prevention",
        "Emergency"
    ]
}

print("\nHIERARCHICAL TAXONOMY")

for category, subcategories in taxonomy.items():
    print(category, ":", subcategories)

# ============================================================
# 3. DOMAIN CONCEPTS AND RELATIONSHIPS
# ============================================================

concepts = {
    "Diabetes": "A condition involving elevated blood glucose.",
    "Blood Glucose": "Amount of glucose in blood.",
    "Insulin": "Hormone regulating blood glucose.",
    "Pancreas": "Organ producing insulin.",
    "HbA1c": "Long-term blood glucose indicator.",
    "Glucose Meter": "Device measuring blood glucose.",
    "Nutrition": "Food patterns affecting health.",
    "Exercise": "Physical activity affecting glucose.",
    "Medication": "Medicines used to manage diabetes.",
    "Hypoglycemia": "Low blood glucose.",
    "Complications": "Long-term health problems.",
    "Healthcare Professional": "Qualified medical professional."
}

relationships = [
    ("Pancreas", "produces", "Insulin"),
    ("Insulin", "regulates", "Blood Glucose"),
    ("Diabetes", "affects", "Blood Glucose"),
    ("Glucose Meter", "measures", "Blood Glucose"),
    ("HbA1c", "indicates", "Blood Glucose"),
    ("Nutrition", "influences", "Blood Glucose"),
    ("Exercise", "affects", "Blood Glucose"),
    ("Medication", "manages", "Diabetes"),
    ("Insulin", "treats", "Diabetes"),
    ("Medication", "may cause", "Hypoglycemia"),
    ("Diabetes", "may cause", "Complications"),
    ("Healthcare Professional", "treats", "Diabetes"),
    ("Healthcare Professional", "orders", "HbA1c"),
    ("Healthcare Professional", "advises on", "Nutrition"),
    ("Hypoglycemia", "is a risk in", "Diabetes")
]

print("\nCONCEPTS:")
for concept, description in concepts.items():
    print(concept, ":", description)

print("\nRELATIONSHIPS:")
for relationship in relationships:
    print(relationship)

# ============================================================
# 4. NLP PREPROCESSING
# ============================================================

stop_words = set(stopwords.words("english")) - {"no", "not", "nor"}

lemmatizer = WordNetLemmatizer()

def preprocess(text):

    # Lowercase conversion
    text = text.lower()

    # Tokenization
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text)

    # Stopword removal
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)


print("\nPREPROCESSING EXAMPLE:")
print(preprocess("DIABETIC patients are eating different foods."))

# ============================================================
# 5. TF-IDF FEATURE EXTRACTION
# ============================================================

X = df["query"]
y = df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    preprocessor=preprocess,
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF TRAINING MATRIX:", X_train_tfidf.shape)

# ============================================================
# 6. LOGISTIC REGRESSION CLASSIFIER
# ============================================================

model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

# ============================================================
# 7. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test, y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test, y_pred,
    average="weighted",
    zero_division=0
)

print("\nMODEL EVALUATION")

print("Accuracy:", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1 Score:", round(f1, 3))

print("\nCLASSIFICATION REPORT")

print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

# Confusion Matrix

labels = sorted(df["intent"].unique())

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

plt.figure(figsize=(10, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Predicted Intent")
plt.ylabel("Actual Intent")
plt.title("Diabetes Assistant Confusion Matrix")
plt.xticks(rotation=40)
plt.tight_layout()
plt.show()

# ============================================================
# 8. DOMAIN-SPECIFIC KNOWLEDGE BASE
# ============================================================

knowledge_base = {

    "symptoms":
    "Common diabetes symptoms include increased thirst, frequent urination, "
    "increased hunger, fatigue, blurred vision and unexplained weight changes. "
    "Symptoms alone cannot confirm diabetes. Consult a healthcare professional.",

    "glucose_monitoring":
    "A glucose meter measures blood glucose at a particular time. "
    "Maintain a dated record of readings and follow the monitoring schedule "
    "recommended by your healthcare team. HbA1c reflects average blood glucose "
    "over approximately 2 to 3 months.",

    "medication":
    "Diabetes medicines help manage blood glucose. Take medicines only as "
    "prescribed. Never independently stop, start or change medication doses. "
    "Contact your doctor or pharmacist regarding missed doses or side effects.",

    "nutrition":
    "A balanced diabetes-friendly eating pattern can include vegetables, "
    "whole grains, fibre-rich foods, lean protein and appropriate portions. "
    "Limit sugary drinks. A dietitian can help personalize your eating plan.",

    "exercise":
    "Walking, cycling and other appropriate physical activities can support "
    "glucose management and cardiovascular health. Start gradually and discuss "
    "exercise precautions with your healthcare professional.",

    "complications":
    "Diabetes may increase the risk of eye, kidney, nerve, foot and heart "
    "problems. Regular eye examinations, kidney tests and foot checks are "
    "important parts of ongoing care.",

    "prevention":
    "Healthy eating, regular physical activity, adequate sleep and maintaining "
    "a healthy weight where appropriate may reduce the risk of type 2 diabetes. "
    "Screening can help identify elevated glucose early.",

    "emergency":
    "MEDICAL EMERGENCY: Contact emergency services immediately. In India, "
    "dial 112. If someone is unconscious, having a seizure or unable to swallow, "
    "do not give anything by mouth. Follow emergency-dispatch instructions."
}

# ============================================================
# 9. RULE-BASED REASONING ENGINE
# ============================================================

emergency_keywords = [
    "unconscious",
    "unresponsive",
    "not responding",
    "collapsed",
    "seizure",
    "cannot swallow",
    "unable to swallow",
    "severe confusion",
    "difficulty breathing",
    "trouble breathing",
    "chest pain",
    "not breathing"
]

diabetes_keywords = [
    "diabetes",
    "diabetic",
    "blood sugar",
    "blood glucose",
    "glucose",
    "insulin",
    "hba1c",
    "hypoglycemia",
    "hyperglycemia",
    "sugar level"
]

CONFIDENCE_THRESHOLD = 0.38


def check_emergency(query):

    query = query.lower()

    for word in emergency_keywords:
        if word in query:
            return True

    return False


def check_scope(query):

    query = query.lower()

    return any(
        word in query
        for word in diabetes_keywords
    )


# ============================================================
# 10. INTEGRATED HEALTHCARE ASSISTANT
# ============================================================

def healthcare_assistant(query):

    query = str(query).strip()

    if not query:
        return "Please enter a question."

    # RULE 1: Emergency escalation

    if check_emergency(query):
        return knowledge_base["emergency"]

    # RULE 2: Unsupported query handling

    if not check_scope(query):

        return (
            "Sorry, I can only answer questions related to "
            "diabetes care and management. Please consult an "
            "appropriate healthcare professional for other concerns."
        )

    # NLP preprocessing and TF-IDF

    processed_query = vectorizer.transform([query])

    # Intent classification

    probabilities = model.predict_proba(processed_query)[0]

    best_index = np.argmax(probabilities)

    intent = model.classes_[best_index]

    confidence = probabilities[best_index]

    # RULE 3: Low confidence handling

    if confidence < CONFIDENCE_THRESHOLD:

        return (
            "I am not confident about understanding your question. "
            "Please rephrase it with more details."
        )

    # RULE 4: Knowledge retrieval

    response = knowledge_base[intent]

    # RULE 5: Safe medical response framing

    return (
        f"Detected Intent: {intent}\n"
        f"Confidence: {confidence:.2f}\n\n"
        f"Response:\n{response}\n\n"
        "Disclaimer: General educational information only. "
        "This assistant does not diagnose or prescribe treatment."
    )


# ============================================================
# 11. TESTING ALL REQUIRED CASES
# ============================================================

test_queries = [

    ("What are common symptoms of diabetes?", "Symptoms"),

    ("How often should I check blood glucose?",
     "Glucose Monitoring"),

    ("What is insulin used for?",
     "Medication"),

    ("What foods are good for diabetes?",
     "Nutrition"),

    ("Can walking reduce blood sugar?",
     "Exercise"),

    ("How does diabetes affect eyes?",
     "Complications"),

    ("How can type 2 diabetes be prevented?",
     "Prevention"),

    ("A diabetic person is unconscious and cannot swallow.",
     "Emergency"),

    ("Why am I thirsty all day and urinating frequently with diabetes?",
     "Differently Worded Query"),

    ("How do I treat malaria at home?",
     "Unsupported Query"),

    ("diabetes xyz blah",
     "Low Confidence Query")
]

print("\n========== TEST RESULTS ==========")

for query, test_type in test_queries:

    print("\nTest:", test_type)

    print("Query:", query)

    print("Response:", healthcare_assistant(query))

    print("-" * 70)


# ============================================================
# 12. GRADIO INTERACTIVE INTERFACE
# ============================================================

def respond(message, history):

    return healthcare_assistant(message)


demo = gr.ChatInterface(
    fn=respond,
    title="Diabetes Care Information Assistant",

    description=(
        "Ask questions about diabetes symptoms, glucose monitoring, "
        "medication, nutrition, exercise, complications and prevention. "
        "This is an educational assistant, not a medical diagnostic tool."
    ),

    examples=[
        "What are common symptoms of diabetes?",
        "How often should I check blood glucose?",
        "What foods are good for diabetes?",
        "How does diabetes affect the eyes?",
        "What is insulin used for?"
    ]
)

print("\nLaunching Diabetes Care Assistant...")

demo.launch(share=True, debug=True)



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\ACER\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'gradio'